# EDA — Twitch Game Pulse

Análisis exploratorio de audiencia de videojuegos en Twitch.

**Datos reales** de la API Helix de Twitch: snapshots del top 100 de juegos tomados entre el 27/08/2026 y el 21/09/2026 (8 días con datos; la ingesta no se ejecutó todos los días).

**Fuente de datos:** Twitch Helix API → `data/twitch_pulse.db`

> Una versión anterior de este análisis mezclaba, sin darse cuenta, 14 días de datos sintéticos (13–26/08, generados con `ingesta.py --sintetico`) con los reales. Esos registros se han eliminado y las conclusiones de abajo están recalculadas solo con datos reales.

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import timedelta
import os

DB_PATH = '../data/twitch_pulse.db'

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f'No se encontró {DB_PATH}. Ejecuta primero: python ingesta.py')

## 1. Carga y vista general

In [2]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql('''
    SELECT s.id, s.juego_id, j.nombre, s.timestamp, s.viewers, s.num_streams
    FROM snapshots_audiencia s
    JOIN juegos j ON s.juego_id = j.id
    ORDER BY s.timestamp
''', conn)

df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
df['fecha'] = df['timestamp'].dt.date

# Periodo analizado: datos reales del 27/08 al 21/09/2026. La ingesta se ha seguido
# ejecutando despues; para ver los datos nuevos usa la app.
FECHA_CORTE = pd.Timestamp('2026-09-22', tz='UTC')
df = df[df['timestamp'] < FECHA_CORTE]

NO_JUEGOS = ['Just Chatting', 'IRL', 'Slots', 'Sports', 'Music', 'Art', 'ASMR', 'Crypto',
             'Talk Shows & Podcasts', 'DJs', 'Poker', 'Animals, Aquariums, and Zoos',
             'Co-working & Studying', 'Pools, Hot Tubs, and Beaches', 'Politics',
             'Software and Game Development', 'Special Events', 'Science & Technology',
             'Games + Demos', 'Retro', 'Education']
df = df[~df['nombre'].isin(NO_JUEGOS)]

print(f'Shape: {df.shape}')
print(f'Rango de fechas: {df.fecha.min()} a {df.fecha.max()}')
print(f'Juegos únicos: {df.nombre.nunique()}')
print(f'Total de snapshots: {len(df)}')
df.head()

Shape: (1001, 7)
Rango de fechas: 2026-08-27 a 2026-09-21
Juegos únicos: 236
Total de snapshots: 1001


,id,juego_id,nombre,timestamp,viewers,num_streams,fecha
1,702,21779,League of Legends,2026-08-27 11:25:24.366183+00:00,120497,1280,2026-08-27
2,703,32399,Counter-Strike,2026-08-27 11:25:32.223429+00:00,101225,1185,2026-08-27
3,704,516575,VALORANT,2026-08-27 11:25:44.172848+00:00,68115,1924,2026-08-27
4,705,27471,Minecraft,2026-08-27 11:25:54.034233+00:00,58637,1737,2026-08-27
6,707,18122,World of Warcraft,2026-08-27 11:26:00.959257+00:00,54617,708,2026-08-27


### Conclusión

El periodo analizado tiene 1.001 snapshots de 236 juegos entre el 27/08/2026 y el 21/09/2026. Cada registro es la audiencia de un juego en un momento dado: viewers, número de streams y hora.

Ojo con dos limitaciones: hay días sin datos y cada snapshot se tomó a una hora distinta (entre las 11:00 y las 20:00 UTC). La audiencia de Twitch cambia mucho a lo largo del día, así que las comparaciones entre fechas tienen ruido.

## 2. Ranking de audiencia — ¿Quién gana la batalla por la atención?

In [3]:
idx_ultimo = df.groupby('nombre')['timestamp'].idxmax()
ultimo = df.loc[idx_ultimo].sort_values('viewers', ascending=False)

top10 = ultimo.head(10)
print('Top 10 juegos por viewers (último snapshot):\n')
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'{i:2d}. {row["nombre"]:30s} | {row["viewers"]:>10,} viewers | {row["num_streams"]:>5} streams')

# Concentración de audiencia
top5_viewers = ultimo.head(5)['viewers'].sum()
total_viewers = ultimo['viewers'].sum()
print(f'\nTop 5 concentran {top5_viewers:,} viewers ({top5_viewers/total_viewers*100:.1f}% del total)')

fig = px.bar(
    top10,
    x='nombre',
    y='viewers',
    color='viewers',
    color_continuous_scale='viridis',
    title='Top 10 juegos por viewers (último snapshot)',
    labels={'nombre': 'Juego', 'viewers': 'Viewers'},
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

Top 10 juegos por viewers (último snapshot):

 1. Grand Theft Auto V             |    157,631 viewers |  3128 streams
 2. World of Warcraft              |    150,974 viewers |  2775 streams
 3. Counter-Strike                 |    111,276 viewers |  2562 streams
 4. Dota 2                         |    103,277 viewers |  1839 streams
 5. League of Legends              |    102,766 viewers |  2465 streams
 6. EA Sports FC 27                |    102,325 viewers |  2329 streams
 7. Minecraft                      |     91,418 viewers |  3339 streams
 8. WARDOGS                        |     81,458 viewers |  2225 streams
 9. VALORANT                       |     79,862 viewers |  3726 streams
10. Fortnite                       |     71,929 viewers |  5602 streams

Top 5 concentran 625,924 viewers (21.5% del total)


### Conclusión

En el último snapshot (21/09/2026), **GTA V** lidera con **157.631 viewers**, seguido de **World of Warcraft** (150.974) y **Counter-Strike** (111.276). Los 5 primeros concentran el **21,5 %** de la audiencia de los juegos rastreados: la atención está bastante repartida entre muchos títulos.

## 3. Evolución temporal — Top 5 juegos

In [4]:
top5_nombres = ultimo.head(5)['nombre'].tolist()
df_top5 = df[df['nombre'].isin(top5_nombres)]

diario = df_top5.groupby(['fecha', 'nombre'])['viewers'].mean().reset_index()
diario['fecha'] = pd.to_datetime(diario['fecha'])

fig = px.line(
    diario,
    x='fecha',
    y='viewers',
    color='nombre',
    title='Evolución diaria de audiencia — Top 5',
    labels={'fecha': 'Fecha', 'viewers': 'Viewers (media diaria)', 'nombre': 'Juego'},
    markers=True,
)
fig.update_layout(legend_title_text='Juego')
fig.show()

print('Estadísticas por juego:')
for nombre in top5_nombres:
    sub = diario[diario['nombre']==nombre]
    print(f'  {nombre}: min={sub["viewers"].min():,.0f}, max={sub["viewers"].max():,.0f}, media={sub["viewers"].mean():,.0f}')

Estadísticas por juego:
  Grand Theft Auto V: min=35,977, max=314,071, media=134,819
  World of Warcraft: min=40,230, max=150,974, media=76,215
  Counter-Strike: min=68,735, max=239,662, media=114,392
  Dota 2: min=38,935, max=103,277, media=59,575
  League of Legends: min=71,101, max=225,300, media=117,653


### Conclusión

La posición en el ranking no cuenta toda la historia. Counter-Strike es el más regular de los grandes (entre 68.735 y 239.662 viewers, media 114.392). GTA V tiene picos muy altos (hasta 314.071) y League of Legends también (225.300). World of Warcraft pasa de unos 40.000–60.000 viewers a 150.974 en el último snapshot. **La posición de un juego y su comportamiento son dos cosas diferentes.**

## 4. Crecimiento — ¿Quién está ganando audiencia?

In [5]:
ahora = df['timestamp'].max()
inicio_actual = ahora - timedelta(days=7)
inicio_anterior = inicio_actual - timedelta(days=7)

semana_actual = df[(df['timestamp'] > inicio_actual) & (df['timestamp'] <= ahora)]
semana_anterior = df[(df['timestamp'] > inicio_anterior) & (df['timestamp'] <= inicio_actual)]

agg_actual = semana_actual.groupby('nombre')['viewers'].mean().reset_index()
agg_actual.columns = ['nombre', 'viewers_actual']
agg_anterior = semana_anterior.groupby('nombre')['viewers'].mean().reset_index()
agg_anterior.columns = ['nombre', 'viewers_anterior']

merged = agg_actual.merge(agg_anterior, on='nombre', how='outer').fillna(0)
merged['crecimiento_pct'] = merged.apply(
    lambda r: ((r['viewers_actual'] - r['viewers_anterior']) / r['viewers_anterior'] * 100)
    if r['viewers_anterior'] > 0 else 0, axis=1
)

top_crec = merged[merged['viewers_anterior'] > 100].sort_values('crecimiento_pct', ascending=False).head(10)
print('Top 10 crecimiento (semana actual vs anterior):\n')
for i, (_, r) in enumerate(top_crec.iterrows(), 1):
    print(f'{i:2d}. {r["nombre"]:30s} | {r["viewers_anterior"]:>8,.0f} → {r["viewers_actual"]:>8,.0f} | {r["crecimiento_pct"]:+6.1f}%')

fig = px.bar(
    top_crec.sort_values('crecimiento_pct', ascending=True),
    x='crecimiento_pct',
    y='nombre',
    orientation='h',
    color='crecimiento_pct',
    color_continuous_scale='RdYlGn',
    title='Top 10 crecimiento de audiencia (%)',
    labels={'nombre': 'Juego', 'crecimiento_pct': 'Crecimiento (%)'},
)
fig.update_layout(showlegend=False, yaxis=dict(autorange='reversed'))
fig.show()

Top 10 crecimiento (semana actual vs anterior):

 1. World of Warcraft              |   43,714 →  150,974 | +245.4%
 2. Phasmophobia                   |    3,351 →   11,340 | +238.4%
 3. Destiny 2                      |    4,519 →   13,421 | +197.0%
 4. Apex Legends                   |   17,558 →   39,234 | +123.5%
 5. PUBG: BATTLEGROUNDS            |   11,108 →   23,203 | +108.9%
 6. Rocket League                  |   18,221 →   32,273 |  +77.1%
 7. Fortnite                       |   40,883 →   71,929 |  +75.9%
 8. Magic: The Gathering           |    3,264 →    5,419 |  +66.0%
 9. Dota 2                         |   67,137 →  103,277 |  +53.8%
10. Teamfight Tactics              |   13,990 →   19,218 |  +37.4%


### Conclusión

Comparando la última semana con la anterior, los que más crecen son **World of Warcraft (+245 %)**, **Phasmophobia (+238 %)** y **Destiny 2 (+197 %)**; entre los grandes también suben Apex Legends (+124 %) y Fortnite (+76 %).

Con tan pocos snapshots por semana, y tomados a horas distintas, estos porcentajes indican dirección, no magnitud exacta. El ranking dice quién está arriba; el crecimiento, hacia dónde se mueve la atención.

## 5. Variabilidad — ¿Todos se comportan igual?

In [6]:
stats = df.groupby('nombre')['viewers'].agg(['mean', 'std', 'max', 'min', 'count']).reset_index()
stats.columns = ['nombre', 'media', 'desv_est', 'maximo', 'minimo', 'count']
stats['cv'] = stats['desv_est'] / stats['media'] * 100
stats = stats[stats['count'] >= 5].sort_values('cv', ascending=False)

top_var = stats.head(10)
print('Top 10 por variabilidad (CV %):\n')
for i, (_, r) in enumerate(top_var.iterrows(), 1):
    print(f'{i:2d}. {r["nombre"]:30s} | CV: {r["cv"]:6.1f}% | Media: {r["media"]:>10,.0f} | Max: {r["maximo"]:>10,}')

fig = px.scatter(
    stats,
    x='media',
    y='cv',
    size='maximo',
    hover_name='nombre',
    title='Variabilidad vs Audiencia media (tamaño = viewers máximos)',
    labels={'media': 'Audiencia media', 'cv': 'Coeficiente de variación (%)'},
)
fig.show()

Top 10 por variabilidad (CV %):

 1. WARDOGS                        | CV:  147.6% | Media:     22,520 | Max:     81,458
 2. Rust                           | CV:  145.4% | Media:     15,199 | Max:     81,834
 3. Hero Siege                     | CV:  116.7% | Media:      9,038 | Max:     24,453
 4. Bodycam                        | CV:  103.5% | Media:     11,444 | Max:     30,331
 5. Star Wars Zero Company         | CV:   94.1% | Media:     11,054 | Max:     25,550
 6. BOMBANANA!                     | CV:   92.6% | Media:     12,053 | Max:     33,324
 7. Halloween: The Game            | CV:   91.7% | Media:     18,833 | Max:     45,764
 8. Rainbow Six Siege              | CV:   89.7% | Media:     17,269 | Max:     60,203
 9. How to Fish                    | CV:   85.0% | Media:      9,076 | Max:     26,841
10. Rocket League                  | CV:   83.5% | Media:     13,728 | Max:     32,273


### Conclusión

Los juegos más variables son **WARDOGS (CV 148 %)**, **Rust (145 %)** y **Hero Siege (117 %)**. Un CV alto indica una audiencia que depende de momentos puntuales (un lanzamiento, un evento o un streamer concreto). Rainbow Six Siege (90 %) es el más inestable entre los títulos consolidados. **Una única métrica puede ocultar información.**

## 6. Eficiencia — Viewers por stream

In [7]:
ultimo_vps = ultimo.copy()
ultimo_vps['viewers_per_stream'] = ultimo_vps['viewers'] / ultimo_vps['num_streams']
top_vps = ultimo_vps.sort_values('viewers_per_stream', ascending=False).head(10)

print('Top 10 viewers por stream (último snapshot):\n')
for i, (_, r) in enumerate(top_vps.iterrows(), 1):
    print(f'{i:2d}. {r["nombre"]:30s} | {r["viewers"]:>8,} viewers | {r["num_streams"]:>5} streams | {r["viewers_per_stream"]:>6.0f} v/s')

fig = px.bar(
    top_vps.sort_values('viewers_per_stream', ascending=True),
    x='viewers_per_stream',
    y='nombre',
    orientation='h',
    color='viewers_per_stream',
    color_continuous_scale='plasma',
    title='Top 10 viewers por stream (eficiencia)',
    labels={'nombre': 'Juego', 'viewers_per_stream': 'Viewers / Stream'},
)
fig.update_layout(showlegend=False)
fig.show()

Top 10 viewers por stream (último snapshot):

 1. BALL BOY Simulator             |   49,212 viewers |     4 streams |  12303 v/s
 2. Piroker                        |   11,574 viewers |     1 streams |  11574 v/s
 3. Kart Racing Pro                |   19,357 viewers |     2 streams |   9678 v/s
 4. X-Men Origins: Wolverine       |   28,126 viewers |     3 streams |   9375 v/s
 5. the cabin game                 |   14,607 viewers |     2 streams |   7304 v/s
 6. Too Many Toys                  |    6,780 viewers |     1 streams |   6780 v/s
 7. Pokémon Brilliant Diamond/Shining Pearl |   43,928 viewers |     7 streams |   6275 v/s
 8. Poppy Playtime                 |   40,456 viewers |     7 streams |   5779 v/s
 9. Unicycle Together              |   15,973 viewers |     3 streams |   5324 v/s
10. Watch Your Plastic Duck        |    9,363 viewers |     2 streams |   4682 v/s


### Conclusión

Juegos como **BALL BOY Simulator** (12.303 viewers por stream con solo 4 streams) o **Poppy Playtime** (5.779 v/s) tienen pocos streams y mucha audiencia: unos pocos creadores concentran toda la atención. Es la señal típica de un juego empujado por un streamer grande más que por la comunidad.

## 7. Síntesis — Las tres perspectivas

Para entender qué ocurre en Twitch no basta con preguntar «¿quién tiene más viewers?». Necesitamos combinar:

| Perspectiva | Qué mide | Ejemplo |
|---|---|---|
| **Volumen** | Cuánta audiencia tiene un juego | GTA V: 157.631 viewers |
| **Evolución** | Si está creciendo o decreciendo | World of Warcraft: +245 % vs semana anterior |
| **Variabilidad** | Si su comportamiento es estable | Rust: CV 145 % |

Un juego puede liderar y ser regular (Counter-Strike), crecer con fuerza (World of Warcraft) o tener una audiencia moderada pero muy irregular (Rust, WARDOGS).